In [1]:
suppressPackageStartupMessages(library(Seurat))
suppressPackageStartupMessages(library(argparse))

In [2]:
here::i_am("processing/1_create_seurat_rna.R")
source(here::here("settings.R"))

# These settings are important for consistency with ArchR, which provides little flexibility to edit cell names
opts$trim.barcode <- FALSE
opts$sample_cell_separator <- "#"

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/12_Eomes_T_Mixl1/T/code



In [62]:
args <- list()
args$inputdir <- paste0(io$basedir,"data")
#args$outdir <- paste0(io$basedir,"/processed/rna")
args$samples <- opts$samples[1:2]

In [63]:
count_mtx <- list()
cell.info <- list()

for (i in args$samples) {
  print(i)
    

  # Load gene metadata
  # gene.loc <- sprintf("%s/%s/genes.tsv.gz",args$inputdir,i)
  # gene.info[[i]] <- fread(gene.loc, header=F) %>%
  #   setnames(c("ens_id","symbol")) %>%
  #   .[,idx:=1:.N]
  # dim(gene.info[[i]])
  
  # Load cell metadata
  barcode.loc <- sprintf("%s/%s/outs/filtered_feature_bc_matrix/barcodes.tsv.gz",args$inputdir,i)
  cell.info[[i]] <- fread(barcode.loc, header=F) %>%
    setnames("barcode") %>%
    .[,barcode:=ifelse(rep(opts$trim.barcode,.N),gsub("-1","",barcode),barcode)] %>%
    .[,c("sample","cell"):=list(i,sprintf("%s%s%s",i,opts$sample_cell_separator,barcode))]
  dim(cell.info[[i]])
  
  # Load matrix  
  count_mtx[[i]] <- Read10X(sprintf("%s/%s/outs/filtered_feature_bc_matrix/",args$inputdir,i))
}

print(lapply(count_mtx,dim))

[1] "sample_1"
[1] "sample_2"
$sample_1
[1] 27995  3481

$sample_2
[1] 27995  2987



In [64]:
#######################
## Keep common genes ##
#######################

genes <- Reduce("intersect",lapply(count_mtx,rownames))
for (i in 1:length(count_mtx)) {
  count_mtx[[i]] <- count_mtx[[i]][genes,]
}

stopifnot(length(unique(lapply(count_mtx,nrow)))==1)
stopifnot(length(unique(lapply(count_mtx,rownames)))==1)

In [65]:
#################
## Concatenate ##
#################

# Concatenate cell metadata
cell.info <- rbindlist(cell.info)
rownames(cell.info) <- cell.info$cell

# Concatenate matrices
count_mtx <- do.call("cbind",count_mtx)
colnames(count_mtx) <- cell.info$cell


In [66]:
# Remove duplicated genes
count_mtx <- count_mtx[!duplicated(rownames(count_mtx)),]

# Sanity checks
stopifnot(sum(duplicated(rownames(count_mtx)))==0)
stopifnot(sum(duplicated(colnames(count_mtx)))==0)


In [67]:
##########################
## Create Seurat object ##
##########################

cell.info.to.seurat <- cell.info[cell%in%colnames(count_mtx)] %>% setkey(cell) %>% .[colnames(count_mtx)] %>% as.data.frame
rownames(cell.info.to.seurat) <- cell.info.to.seurat$cell
stopifnot(rownames(cell.info.to.seurat)==colnames(count_mtx))
stopifnot(sum(is.na(rownames(cell.info.to.seurat$cell)))==0)

In [68]:
head(cell.info.to.seurat)

,barcode,sample,cell
,<chr>,<chr>,<chr>
sample_1#AAACCTGAGAAAGTGG,AAACCTGAGAAAGTGG,sample_1,sample_1#AAACCTGAGAAAGTGG
sample_1#AAACCTGAGGTGCAAC,AAACCTGAGGTGCAAC,sample_1,sample_1#AAACCTGAGGTGCAAC
sample_1#AAACCTGCAATACGCT,AAACCTGCAATACGCT,sample_1,sample_1#AAACCTGCAATACGCT
sample_1#AAACCTGCAGCTTAAC,AAACCTGCAGCTTAAC,sample_1,sample_1#AAACCTGCAGCTTAAC
sample_1#AAACCTGCAGTACACT,AAACCTGCAGTACACT,sample_1,sample_1#AAACCTGCAGTACACT
sample_1#AAACCTGGTCAATACC,AAACCTGGTCAATACC,sample_1,sample_1#AAACCTGGTCAATACC


In [69]:
tail(rownames(count_mtx))
head(colnames(count_mtx))

[1] "mt-Nd4l"   "mt-Nd4"    "mt-Nd5"    "mt-Nd6"    "mt-Cytb"   "tomato-td"

[1] "sample_1#AAACCTGAGAAAGTGG" "sample_1#AAACCTGAGGTGCAAC"
[3] "sample_1#AAACCTGCAATACGCT" "sample_1#AAACCTGCAGCTTAAC"
[5] "sample_1#AAACCTGCAGTACACT" "sample_1#AAACCTGGTCAATACC"

In [70]:
test = grep("[^[:alnum:]]",genes, value = TRUE)
test

[1] "Krtap28-10"    "Krtap28-13"    "Zfp813-ps"     "Olfr364-ps1"  
  [5] "Olfr1025-ps1"  "Olfr1117-ps1"  "Olfr1150-ps1"  "Olfr1192-ps1" 
  [9] "Olfr1185-ps1"  "Olfr1191-ps1"  "Olfr1224-ps1"  "Olfr1267-ps1" 
 [13] "Olfr1274-ps"   "Olfr1315-ps1"  "Nkx2-4"        "Nkx2-2"       
 [17] "Nkx2-2os"      "Spin2-ps6"     "Btg1-ps1"      "Btg1-ps2"     
 [21] "Olfr1326-ps1"  "Mir124-2hg"    "Fam188b2-ps"   "Sprr2j-ps"    
 [25] "Atg4a-ps"      "Vma21-ps"      "Nkx1-1"        "Nkx3-2"       
 [29] "Nkx6-1"        "Smkr-ps"       "Olfr237-ps1"   "Gt(ROSA)26Sor"
 [33] "Speer9-ps1"    "Rasl2-9"       "Zscan4-ps1"    "Zscan4-ps2"   
 [37] "Zscan4-ps3"    "Obox4-ps35"    "Mir9-3hg"      "Olfr548-ps1"  
 [41] "Olfr573-ps1"   "Olfr588-ps1"   "Olfr625-ps1"   "Hbb-bt"       
 [45] "Hbb-bs"        "Hbb-bh2"       "Hbb-bh1"       "Hbb-y"        
 [49] "Olfr1532-ps1"  "Olfr709-ps1"   "Nkx1-2"        "Nkx6-2"       
 [53] "Krtap5-2"      "Krtap5-3"      "Krtap5-5"      "Krtap5-1"     
 [57] "Krtap5-4"      "Krtap12-1"     "Krtap10-4"     "Anapc15-ps"   
 [61] "Nkx6-3"        "Rpl21-ps4"     "Olfr721-ps1"   "Rpl23a-ps3"   
 [65] "Vmn2r-ps111"   "Rpl13-ps3"     "Mir124a-1hg"   "Nkx2-6"       
 [69] "Nkx3-1"        "Rps2-ps6"      "Tpm3-rs7"      "Olfr839-ps1"  
 [73] "Olfr896-ps1"   "Olfr911-ps1"   "Rpl10-ps3"     "Hba-x"        
 [77] "Hba-a1"        "Hba-a2"        "Olfr329-ps"    "Olfr391-ps"   
 [81] "Rpl9-ps1"      "Krtap3-3"      "Krtap3-2"      "Krtap3-1"     
 [85] "Krtap1-5"      "Krtap1-4"      "Krtap1-3"      "Krtap9-3"     
 [89] "Krtap2-4"      "Krtap4-1"      "Krtap4-2"      "Krtap4-7"     
 [93] "Krtap4-6"      "Krtap4-8"      "Krtap4-9"      "Krtap4-13"    
 [97] "Krtap4-16"     "Krtap9-1"      "Krtap31-1"     "Krtap31-2"    
[101] "Krtap9-5"      "Krtap29-1"     "Krtap16-1"     "Krtap17-1"    
[105] "Hmga1-rs1"     "Tex19.2"       "Tex19.1"       "Olfr1369-ps1" 
[109] "Olfr465-ps1"   "Vmn2r-ps104"   "Ftl1-ps1"      "Apoo-ps"      
[113] "Nkx2-1"        "Nkx2-9"        "Fam136b-ps"    "Rpl7a-ps3"    
[117] "Cyp2d37-ps"    "Olfr175-ps1"   "Krtap24-1"     "Krtap26-1"    
[121] "Krtap27-1"     "Krtap13-1"     "Krtap19-1"     "Krtap19-2"    
[125] "Krtap19-3"     "Krtap19-4"     "Krtap19-5"     "Krtap19-9a"   
[129] "Krtap19-9b"    "Krtap16-3"     "Krtap22-2"     "Krtap6-1"     
[133] "Krtap6-5"      "Krtap6-3"      "Krtap20-2"     "Krtap21-1"    
[137] "Krtap6-2"      "Krtap8-1"      "Krtap7-1"      "Krtap11-1"    
[141] "Fpr-rs4"       "Fpr-rs7"       "Fpr-rs6"       "Fpr-rs3"      
[145] "Vmn2r-ps130"   "Nkx2-5"        "H2-K1"         "H2-Ke6"       
[149] "H2-Oa"         "H2-DMa"        "H2-DMb2"       "H2-DMb1"      
[153] "H2-Ob"         "H2-Ab1"        "H2-Aa"         "H2-Eb1"       
[157] "H2-Eb2"        "H2-D1"         "H2-Q1"         "H2-Q2"        
[161] "H2-Q4"         "H2-Q6"         "H2-Q7"         "H2-Q10"       
[165] "H2-T24"        "H2-T23"        "H2-T22"        "H2-T3"        
[169] "H2-M10.2"      "H2-M10.1"      "H2-M10.3"      "H2-M10.4"     
[173] "H2-M11"        "H2-M9"         "H2-M1"         "H2-M10.5"     
[177] "H2-M10.6"      "H2-M5"         "H2-M3"         "H2-M2"        
[181] "Rpl7a-ps5"     "Rpl36-ps4"     "Rpl27-ps3"     "Mir133a-1hg"  
[185] "Pcna-ps2"      "Olfr1555-ps1"  "Olfr1438-ps1"  "Olfr1493-ps1" 
[189] "Rpl9-ps6"      "Nkx2-3"        "Rpl13a-ps1"    "Nutf2-ps1"    
[193] "Rps12-ps3"     "mt-Nd1"        "mt-Nd2"        "mt-Co1"       
[197] "mt-Co2"        "mt-Atp8"       "mt-Atp6"       "mt-Co3"       
[201] "mt-Nd3"        "mt-Nd4l"       "mt-Nd4"        "mt-Nd5"       
[205] "mt-Nd6"        "mt-Cytb"       "tomato-td"

In [71]:
test

[1] "Krtap28-10"    "Krtap28-13"    "Zfp813-ps"     "Olfr364-ps1"  
  [5] "Olfr1025-ps1"  "Olfr1117-ps1"  "Olfr1150-ps1"  "Olfr1192-ps1" 
  [9] "Olfr1185-ps1"  "Olfr1191-ps1"  "Olfr1224-ps1"  "Olfr1267-ps1" 
 [13] "Olfr1274-ps"   "Olfr1315-ps1"  "Nkx2-4"        "Nkx2-2"       
 [17] "Nkx2-2os"      "Spin2-ps6"     "Btg1-ps1"      "Btg1-ps2"     
 [21] "Olfr1326-ps1"  "Mir124-2hg"    "Fam188b2-ps"   "Sprr2j-ps"    
 [25] "Atg4a-ps"      "Vma21-ps"      "Nkx1-1"        "Nkx3-2"       
 [29] "Nkx6-1"        "Smkr-ps"       "Olfr237-ps1"   "Gt(ROSA)26Sor"
 [33] "Speer9-ps1"    "Rasl2-9"       "Zscan4-ps1"    "Zscan4-ps2"   
 [37] "Zscan4-ps3"    "Obox4-ps35"    "Mir9-3hg"      "Olfr548-ps1"  
 [41] "Olfr573-ps1"   "Olfr588-ps1"   "Olfr625-ps1"   "Hbb-bt"       
 [45] "Hbb-bs"        "Hbb-bh2"       "Hbb-bh1"       "Hbb-y"        
 [49] "Olfr1532-ps1"  "Olfr709-ps1"   "Nkx1-2"        "Nkx6-2"       
 [53] "Krtap5-2"      "Krtap5-3"      "Krtap5-5"      "Krtap5-1"     
 [57] "Krtap5-4"      "Krtap12-1"     "Krtap10-4"     "Anapc15-ps"   
 [61] "Nkx6-3"        "Rpl21-ps4"     "Olfr721-ps1"   "Rpl23a-ps3"   
 [65] "Vmn2r-ps111"   "Rpl13-ps3"     "Mir124a-1hg"   "Nkx2-6"       
 [69] "Nkx3-1"        "Rps2-ps6"      "Tpm3-rs7"      "Olfr839-ps1"  
 [73] "Olfr896-ps1"   "Olfr911-ps1"   "Rpl10-ps3"     "Hba-x"        
 [77] "Hba-a1"        "Hba-a2"        "Olfr329-ps"    "Olfr391-ps"   
 [81] "Rpl9-ps1"      "Krtap3-3"      "Krtap3-2"      "Krtap3-1"     
 [85] "Krtap1-5"      "Krtap1-4"      "Krtap1-3"      "Krtap9-3"     
 [89] "Krtap2-4"      "Krtap4-1"      "Krtap4-2"      "Krtap4-7"     
 [93] "Krtap4-6"      "Krtap4-8"      "Krtap4-9"      "Krtap4-13"    
 [97] "Krtap4-16"     "Krtap9-1"      "Krtap31-1"     "Krtap31-2"    
[101] "Krtap9-5"      "Krtap29-1"     "Krtap16-1"     "Krtap17-1"    
[105] "Hmga1-rs1"     "Tex19.2"       "Tex19.1"       "Olfr1369-ps1" 
[109] "Olfr465-ps1"   "Vmn2r-ps104"   "Ftl1-ps1"      "Apoo-ps"      
[113] "Nkx2-1"        "Nkx2-9"        "Fam136b-ps"    "Rpl7a-ps3"    
[117] "Cyp2d37-ps"    "Olfr175-ps1"   "Krtap24-1"     "Krtap26-1"    
[121] "Krtap27-1"     "Krtap13-1"     "Krtap19-1"     "Krtap19-2"    
[125] "Krtap19-3"     "Krtap19-4"     "Krtap19-5"     "Krtap19-9a"   
[129] "Krtap19-9b"    "Krtap16-3"     "Krtap22-2"     "Krtap6-1"     
[133] "Krtap6-5"      "Krtap6-3"      "Krtap20-2"     "Krtap21-1"    
[137] "Krtap6-2"      "Krtap8-1"      "Krtap7-1"      "Krtap11-1"    
[141] "Fpr-rs4"       "Fpr-rs7"       "Fpr-rs6"       "Fpr-rs3"      
[145] "Vmn2r-ps130"   "Nkx2-5"        "H2-K1"         "H2-Ke6"       
[149] "H2-Oa"         "H2-DMa"        "H2-DMb2"       "H2-DMb1"      
[153] "H2-Ob"         "H2-Ab1"        "H2-Aa"         "H2-Eb1"       
[157] "H2-Eb2"        "H2-D1"         "H2-Q1"         "H2-Q2"        
[161] "H2-Q4"         "H2-Q6"         "H2-Q7"         "H2-Q10"       
[165] "H2-T24"        "H2-T23"        "H2-T22"        "H2-T3"        
[169] "H2-M10.2"      "H2-M10.1"      "H2-M10.3"      "H2-M10.4"     
[173] "H2-M11"        "H2-M9"         "H2-M1"         "H2-M10.5"     
[177] "H2-M10.6"      "H2-M5"         "H2-M3"         "H2-M2"        
[181] "Rpl7a-ps5"     "Rpl36-ps4"     "Rpl27-ps3"     "Mir133a-1hg"  
[185] "Pcna-ps2"      "Olfr1555-ps1"  "Olfr1438-ps1"  "Olfr1493-ps1" 
[189] "Rpl9-ps6"      "Nkx2-3"        "Rpl13a-ps1"    "Nutf2-ps1"    
[193] "Rps12-ps3"     "mt-Nd1"        "mt-Nd2"        "mt-Co1"       
[197] "mt-Co2"        "mt-Atp8"       "mt-Atp6"       "mt-Co3"       
[201] "mt-Nd3"        "mt-Nd4l"       "mt-Nd4"        "mt-Nd5"       
[205] "mt-Nd6"        "mt-Cytb"       "tomato-td"

In [72]:
seurat <- CreateSeuratObject(counts = count_mtx, meta.data = cell.info.to.seurat)